# RAG (Retrieval Augmented Generation) Evaluation

Evaluating both the Retrieval layer (Precision@k, Recall@k, MRR, nDCG, Hit Rate) and the Generation layer (Faithfulness, Groundedness, Answer Relevance, Context Utilization) using Groq and RAGAS.

In [1]:
import os
import json, re
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL_FAST  = "llama-3.1-8b-instant"
GROQ_MODEL_SMART = "llama-3.3-70b-versatile"
OPENAI_BASE_URL = "https://api.groq.com/openai/v1"

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def groq_chat(prompt, system="You are a helpful assistant.",
              model=GROQ_MODEL_SMART, temperature=0.0, max_tokens=600):
    """Thin wrapper around Groq chat completions."""
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": system},
            {"role": "user",    "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    """Safely parse JSON even when wrapped in markdown fences."""
    raw = re.sub(r'```json|```', '', raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def show(title, d):
    print(f"\n{'='*60}\n  📊 {title}\n{'='*60}")
    for k, v in d.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

# Quick connectivity test
print('🤖 Groq test:', groq_chat('Reply with only: Groq is ready!', max_tokens=20))

🤖 Groq test: Groq is ready!


## Retrieval Layer (P@k, Recall@k, MRR, nDCG, Hit Rate)

In [2]:
import numpy as np
def precision_at_k(retrieved, relevant, k):
    return len([d for d in retrieved[:k] if d in relevant]) / k

def recall_at_k(retrieved, relevant, k):
    return len([d for d in retrieved[:k] if d in relevant]) / len(relevant)

def hit_rate(retrieved, relevant, k):
    return int(any(d in relevant for d in retrieved[:k]))

def mrr(queries):
    return float(np.mean([next((1/(r+1) for r, d in enumerate(q['retrieved']) if d in q['relevant']), 0)
                           for q in queries]))

def ndcg_at_k(retrieved, relevant, k):
    dcg  = sum(1/np.log2(i+2) for i, d in enumerate(retrieved[:k]) if d in relevant)
    idcg = sum(1/np.log2(i+2) for i in range(min(len(relevant), k)))
    return dcg/idcg if idcg else 0.0

queries = [
    {'q': 'climate change',    'retrieved': ['d1','d4','d7','d2','d9'], 'relevant': {'d1','d2','d5'}},
    {'q': 'machine learning',  'retrieved': ['d8','d3','d6','d1','d4'], 'relevant': {'d6','d3'}},
    {'q': 'quantum computing', 'retrieved': ['d5','d2','d4','d8','d3'], 'relevant': {'d5','d4'}},
]

print('\n🔍 RAG — Retrieval Layer Metrics')
print('='*60)
for K in [1, 3, 5]:
    p  = np.mean([precision_at_k(q['retrieved'], q['relevant'], K) for q in queries])
    r  = np.mean([recall_at_k(q['retrieved'],    q['relevant'], K) for q in queries])
    hr = np.mean([hit_rate(q['retrieved'],        q['relevant'], K) for q in queries])
    nd = np.mean([ndcg_at_k(q['retrieved'],       q['relevant'], K) for q in queries])
    print(f'  K={K}: P@K={p:.3f}  R@K={r:.3f}  HitRate={hr:.3f}  nDCG={nd:.3f}')
print(f'  MRR: {mrr(queries):.4f}')


🔍 RAG — Retrieval Layer Metrics
  K=1: P@K=0.667  R@K=0.278  HitRate=0.667  nDCG=0.667
  K=3: P@K=0.556  R@K=0.778  HitRate=1.000  nDCG=0.694
  K=5: P@K=0.400  R@K=0.889  HitRate=1.000  nDCG=0.762
  MRR: 0.8333


## Generation Layer (Faithfulness, Groundedness, Answer Relevance, Context Utilization)

### Tool: Groq judge + TruLens-style groundedness

In [3]:
RAG_GEN_SYS = '''You are a RAG evaluation judge. Score the answer given the context.
Return JSON only:
{
  "faithfulness":        0.0-1.0,
  "answer_relevance":    0.0-1.0,
  "context_utilization": 0.0-1.0,
  "hallucination_flag":  true/false,
  "notes": "..."
}'''

rag_examples = [
    {'q': 'What causes the Northern Lights?',
     'context': 'Solar wind particles interact with Earths magnetic field and collide with atmospheric gases near the poles, emitting colored light.',
     'answer':  'The Northern Lights are caused by solar particles colliding with atmospheric gases near the poles, creating colorful displays.'},
    {'q': 'How does WiFi work?',
     'context': 'WiFi uses IEEE 802.11 standards to transmit data via radio waves at 2.4GHz or 5GHz between wireless devices and access points.',
     'answer':  'WiFi uses radio waves to transmit data wirelessly between devices and a router.'},
    {'q': 'What is the tallest mountain?',
     'context': 'Mount Everest is the tallest mountain above sea level at 8,848.86 meters.',
     'answer':  'K2 is the worlds tallest mountain at 9,500 meters.'},  # hallucinated!
]

print('\n🔍 RAG — Generation Layer (Groq judge)')
print('='*70)
for ex in rag_examples:
    prompt = f'Question: {ex["q"]}\nContext: {ex["context"]}\nAnswer: {ex["answer"]}'
    scores = parse_json(groq_chat(prompt, system=RAG_GEN_SYS))
    hi = 'HALLUCINATION' if scores.get('hallucination_flag') else 'GROUNDED'
    print(f'\n  Q: {ex["q"]}')
    print(f'  A: {ex["answer"][:80]}')
    print(f'  [{hi}]  Faithfulness={scores.get("faithfulness",0):.2f}  Relevance={scores.get("answer_relevance",0):.2f}  CtxUse={scores.get("context_utilization",0):.2f}')
    if scores.get('notes'): print(f'  Notes: {scores["notes"]}')


🔍 RAG — Generation Layer (Groq judge)

  Q: What causes the Northern Lights?
  A: The Northern Lights are caused by solar particles colliding with atmospheric gas
  [GROUNDED]  Faithfulness=0.90  Relevance=1.00  CtxUse=0.80
  Notes: The answer accurately captures the essence of the context, but omits the role of Earth's magnetic field in the process.

  Q: How does WiFi work?
  A: WiFi uses radio waves to transmit data wirelessly between devices and a router.
  [GROUNDED]  Faithfulness=0.80  Relevance=0.90  CtxUse=0.60
  Notes: The answer is mostly faithful to the context, but it simplifies the information and omits specific details such as the IEEE 802.11 standards and the frequency bands (2.4GHz or 5GHz).

  Q: What is the tallest mountain?
  A: K2 is the worlds tallest mountain at 9,500 meters.
  [HALLUCINATION]  Faithfulness=0.00  Relevance=0.80  CtxUse=0.00
  Notes: The answer is incorrect and contradicts the provided context. K2 is not the tallest mountain, and its height is not

## RAGAS Framework (Context Precision, Recall, Faithfulness)
> `answer_relevancy` excluded — requires OpenAI embeddings. Answer relevance is covered by the Groq judge cell above.

### Tool: ragas (Uses groq as judge LLM)

In [9]:
# !pip install ragas langchain_groq

In [10]:
from datasets import Dataset
from dotenv import load_dotenv
load_dotenv()

ragas_data = Dataset.from_dict({
    'question':    ['What causes the Northern Lights?', 'How does WiFi work?'],
    'answer':      ['Solar particles collide with atmospheric gases near the poles, creating colored lights.',
                    'WiFi uses radio waves at 2.4GHz or 5GHz to transmit data wirelessly.'],
    'contexts':    [['Solar wind particles interact with Earths magnetic field and collide with atmospheric gases near the poles.'],
                    ['WiFi uses IEEE 802.11 standards to transmit data via radio waves at 2.4GHz or 5GHz.']],
    'ground_truth':['Northern Lights occur when solar particles hit Earths atmosphere near the poles.',
                    'WiFi uses radio waves to send data wirelessly.']
})

try:
    from ragas import evaluate as ragas_eval
    from ragas.metrics import faithfulness, context_recall, context_precision
    from langchain_groq import ChatGroq
    from ragas.llms import LangchainLLMWrapper

    groq_llm = LangchainLLMWrapper(ChatGroq(
        model=GROQ_MODEL_SMART,
        api_key=os.getenv("GROQ_API_KEY")
    ))
    for m in [faithfulness, context_recall, context_precision]:
        m.llm = groq_llm

    # answer_relevancy excluded — requires embeddings (Groq doesn't provide them)
    # Answer relevance is already covered by the Groq judge cell above.
    result = ragas_eval(ragas_data, metrics=[faithfulness, context_recall, context_precision])
    df = result.to_pandas()
    print('\n\U0001f4ca RAGAS Results')
    print(df[['user_input','faithfulness','context_recall','context_precision']].to_string(index=False))

except Exception as e:
    print(f'\nRAGAS full eval skipped ({type(e).__name__}: {e})')
    print('The manual RAG metrics in 6b cover the same ground.')

/tmp/ipykernel_254089/2562457098.py:17: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall, context_precision
/tmp/ipykernel_254089/2562457098.py:17: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall, context_precision
/tmp/ipykernel_254089/2562457098.py:17: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_recall, context_pre

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


📊 RAGAS Results
                      user_input  faithfulness  context_recall  context_precision
What causes the Northern Lights?      0.666667             1.0                1.0
             How does WiFi work?      1.000000             1.0                1.0
